# Stage GRCh38 repetitive-context tracks

One-time Terra / Workbench notebook to download UCSC hg38 `rmsk`, `simpleRepeat`,
and `genomicSuperDups`, convert them to sorted 3-column BED (chrom / start / end,
0-based, half-open), bgzip them, and upload to the workspace bucket for:

- `AnnotateSvCallset.rmsk_bed`
- `AnnotateSvCallset.simple_repeat_bed`
- `AnnotateSvCallset.segdup_bed`

A site is **repetitive** if it overlaps the union of these three tracks
(`sv_annotation/docs/definitions.md`).

**Requirements**
- A few GB free disk (`rmsk` is the largest dump)
- Network access to `hgdownload.soe.ucsc.edu`
- `gsutil` / workspace bucket write access
- `bgzip` + `tabix` if available (falls back to gzip)

Run this **before** submitting `AnnotateSvCallset`. `sv_03_manuscript_stats.ipynb`
is post-workflow.

In [ ]:
from __future__ import annotations

import gzip
import json
import os
import shutil
import subprocess
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

UCSC_DB = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database"
MIN_FREE_GB = 5

# UCSC dump column indices (0-based). Confirmed against hg38 *.sql table defs:
# rmsk: genoName/genoStart/genoEnd; simpleRepeat + genomicSuperDups: chrom/chromStart/chromEnd.
TRACKS = [
    {
        "name": "rmsk",
        "url": f"{UCSC_DB}/rmsk.txt.gz",
        "chrom_col": 5,
        "start_col": 6,
        "end_col": 7,
        "out_name": "rmsk.bed.gz",
        "wdl_input": "AnnotateSvCallset.rmsk_bed",
    },
    {
        "name": "simpleRepeat",
        "url": f"{UCSC_DB}/simpleRepeat.txt.gz",
        "chrom_col": 1,
        "start_col": 2,
        "end_col": 3,
        "out_name": "simpleRepeat.bed.gz",
        "wdl_input": "AnnotateSvCallset.simple_repeat_bed",
    },
    {
        "name": "genomicSuperDups",
        "url": f"{UCSC_DB}/genomicSuperDups.txt.gz",
        "chrom_col": 1,
        "start_col": 2,
        "end_col": 3,
        "out_name": "genomicSuperDups.bed.gz",
        "wdl_input": "AnnotateSvCallset.segdup_bed",
    },
]

WORK = Path(os.environ.get("REPEAT_TRACK_WORK", Path.cwd() / "repeat_track_work")).resolve()
RAW = WORK / "ucsc_raw"
BEDS = WORK / "beds"

WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
GCS_PREFIX = os.environ.get(
    "REPEAT_TRACK_GCS_PREFIX",
    f"{WORKSPACE_BUCKET}/refs/grch38" if WORKSPACE_BUCKET else "",
)

print("WORK:", WORK)
print("GCS_PREFIX:", GCS_PREFIX or "(set WORKSPACE_BUCKET or REPEAT_TRACK_GCS_PREFIX)")

In [ ]:
def run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("+", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n")
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(
            proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr
        )
    return proc


for d in (WORK, RAW, BEDS):
    d.mkdir(parents=True, exist_ok=True)

available = shutil.disk_usage(WORK).free / (1024 ** 3)
print(f"Free disk at {WORK}: {available:.1f} GB")
if available < MIN_FREE_GB:
    raise SystemExit(
        f"Need ≥{MIN_FREE_GB} GB free to stage UCSC dumps; have {available:.1f} GB."
    )

HAVE_BGZIP = shutil.which("bgzip") is not None
HAVE_TABIX = shutil.which("tabix") is not None
print("bgzip:", HAVE_BGZIP, "tabix:", HAVE_TABIX)

In [ ]:
def download(url: str, dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Reusing {dest}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".partial")
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(dest)
    print(f"Wrote {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


for track in TRACKS:
    track["raw"] = download(track["url"], RAW / f"{track['name']}.txt.gz")

In [ ]:
def dump_to_bed(src: Path, dest_unsorted: Path, chrom_col: int, start_col: int, end_col: int) -> int:
    n = 0
    with gzip.open(src, "rt") as fh, dest_unsorted.open("w") as out:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            needed = max(chrom_col, start_col, end_col)
            if len(parts) <= needed:
                continue
            chrom = parts[chrom_col].strip()
            if not chrom:
                continue
            start = int(parts[start_col])
            end = int(parts[end_col])
            if end < start:
                start, end = end, start
            if end == start:
                end = start + 1
            out.write(f"{chrom}\t{start}\t{end}\n")
            n += 1
    return n


def compress_bed(sorted_bed: Path, out_gz: Path) -> None:
    if out_gz.exists():
        out_gz.unlink()
    tbi = Path(str(out_gz) + ".tbi")
    if tbi.exists():
        tbi.unlink()
    if HAVE_BGZIP:
        with out_gz.open("wb") as fh:
            print("+ bgzip -c", sorted_bed)
            subprocess.run(["bgzip", "-c", str(sorted_bed)], stdout=fh, check=True)
        if HAVE_TABIX:
            run(["tabix", "-p", "bed", str(out_gz)])
    else:
        with sorted_bed.open("rb") as src, gzip.open(out_gz, "wb") as dest:
            shutil.copyfileobj(src, dest)


manifest = {
    "genome_build": "GRCh38",
    "source": UCSC_DB,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "tracks": [],
}

for track in TRACKS:
    unsorted = BEDS / f"{track['name']}.unsorted.bed"
    sorted_bed = BEDS / f"{track['name']}.bed"
    out_gz = BEDS / track["out_name"]
    n = dump_to_bed(
        track["raw"],
        unsorted,
        track["chrom_col"],
        track["start_col"],
        track["end_col"],
    )
    print(f"{track['name']}: {n:,} intervals")
    if n == 0:
        raise SystemExit(f"No intervals parsed from {track['raw']}")
    run(["sort", "-k1,1", "-k2,2n", "-o", str(sorted_bed), str(unsorted)])
    compress_bed(sorted_bed, out_gz)
    track["bed_gz"] = out_gz
    track["n_intervals"] = n
    print(f"Wrote {out_gz} ({out_gz.stat().st_size / 1e6:.1f} MB)")
    unsorted.unlink(missing_ok=True)
    sorted_bed.unlink(missing_ok=True)
    manifest["tracks"].append(
        {
            "name": track["name"],
            "source_url": track["url"],
            "n_intervals": n,
            "file": track["out_name"],
        }
    )

MANIFEST = BEDS / "repeat_tracks.manifest.json"
MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")
print("manifest:", MANIFEST)

In [ ]:
if not GCS_PREFIX:
    raise SystemExit(
        "Set WORKSPACE_BUCKET (Terra default) or REPEAT_TRACK_GCS_PREFIX before uploading."
    )

uploaded: dict[str, str] = {}
for track in TRACKS:
    dest = f"{GCS_PREFIX}/{track['out_name']}"
    print(f"Uploading {track['bed_gz']} -> {dest}")
    run(["gsutil", "-m", "cp", str(track["bed_gz"]), dest])
    tbi = Path(str(track["bed_gz"]) + ".tbi")
    if tbi.exists():
        run(["gsutil", "-m", "cp", str(tbi), dest + ".tbi"])
    uploaded[track["wdl_input"]] = dest

run(["gsutil", "-m", "cp", str(MANIFEST), f"{GCS_PREFIX}/repeat_tracks.manifest.json"])
run(["gsutil", "ls", "-lh", f"{GCS_PREFIX}/"])

print("\nUse these WDL inputs:")
for key, uri in uploaded.items():
    print(f'  "{key}": "{uri}",')

## Optional cleanup

After a successful upload you can delete the local work directory to reclaim disk:

In [ ]:
# shutil.rmtree(WORK)
# print("Removed", WORK)